In [ ]:
import os
from os.path import join
import pickle
import pandas as pd
import mne
from mne.channels import find_ch_adjacency
from mne.datasets import sample
from mne.stats import combine_adjacency, f_mway_rm, f_threshold_mway_rm, spatio_temporal_cluster_test
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
## modify the test variables here
hemi = 'lh'
use_roi = True # subset stcs with spatial_exclude in test
search_tmin = 0
search_tmax = 400

In [ ]:
expt = 'EXPT'
ROOT = f'/path/to/{expt}'
os.chdir(ROOT)

subjects_dir = join(ROOT,'mri')
log_dir  = join(ROOT, 'logs')
stc_dir = join(ROOT, 'stc')
stats_dir = join(ROOT, 'stats')

excluded = ['R0000']
subjects = [i[:5] for i in os.listdir(subjects_dir) if i.startswith('R') and i[:5] not in excluded and not i.endswith(".fif")]

output_dir = os.path.join(ROOT, 'figures')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [ ]:
# brodmann areas of interest
brodmann_areas = ['20', '21', '22', '38', '39', '11', '44', '45']

# define colors for conditions
colors = {
    'CONDITION1': 'tab:blue',
    'CONDITION2': 'tab:orange',
    'CONDITION3': 'tab:green',
    'CONDITION4': 'tab:red'
}

conditions = list(colors.keys())

In [ ]:
def get_STC(subj, cond):
    stc_fname = os.path.join(stc_dir, '%s', '%s_%s_dSPM') % (cond, subj, cond)
    stc = mne.read_source_estimate(stc_fname, subject='fsaverage')
    return stc

def stat_fun(*args):
    return f_mway_rm(np.swapaxes(args, 1, 0), factor_levels=factor_levels,
                     effects=effects, return_pvals=True)[0]

In [ ]:
### set up source space (fsaverage)
src_fname = os.path.join(subjects_dir, 'fsaverage', 'bem', 'fsaverage-ico-4-src.fif')
src = mne.read_source_spaces(src_fname)

In [ ]:
### set up ROI
# this gives you a Brodmann area label as a subset, but more options can be found in /mri/fsaverage/label
num = brodmann_areas[0]

## subset by brodmann (as an example):
annot_name = 'PALS_B12_Brodmann'
brodmann = mne.read_labels_from_annot('fsaverage', annot_name, subjects_dir = subjects_dir, hemi = hemi)
label_name = f'Brodmann.{num}-{hemi}'
roi = [label for label in brodmann if label.name==label_name][0] # you can also add labels for a larger roi

# look for vertex indices in the ROIs 
n_hemisources = 2562
hemi_idx      = np.arange(0, n_hemisources, 1)
roi_idx       = roi.get_vertices_used(vertices=hemi_idx)
diff_idx      = np.setdiff1d(hemi_idx, roi_idx)

spatial_exclude = diff_idx if use_roi else None

In [ ]:
### you can plot this here
## visualize parcellation with fsaverage
Brain = mne.viz.get_brain_class()

brain = Brain(
    "fsaverage",
    hemi,
    "pial",  # Use the pial surface
    subjects_dir=subjects_dir,
    cortex="low_contrast",
    background="white",
    size=(800, 600),
    views=['lateral']
)
        
brain.add_label(roi, borders=False, color='tab:purple')

In [ ]:
# load in data and subset by search_tmin, search_tmax
data_time = []
data = []

for c, cond in enumerate(conditions):
    times = np.arange(-100, 801, 1)
    n_subj = len(subjects)
    data_tmp = np.empty((n_subj, n_hemisources, len(times)))
    print(f'Reading in STCs for condition {cond}')
    
    for s, subj in enumerate(subjects):
        stc = get_STC(subj, cond)
        data_tmp[s,:,:] = stc.data[:n_hemisources]
    
    data.append(np.transpose(data_tmp, [0, 2, 1]))
    
    tmin_idx = np.squeeze(np.where(times==search_tmin))
    tmax_idx = np.squeeze(np.where(times==search_tmax))
    toi_idx = np.arange(tmin_idx, tmax_idx+1)
    data_time.append(data[c][:,toi_idx,:n_hemisources])
    
print(f"Time window of analysis: {search_tmin} to {search_tmax}")

In [ ]:
#---permutation test vars
tail = 0 # signed data
p_thresh = 0.05
n_permutations = 50 # we usually use 10000

factor_levels = [2,2] # 2x2 ANOVA is [2,2]
effects = 'A' # A - main effect of A, B - main effect of B, A:B - interaction effect
f_thresh = f_threshold_mway_rm(n_subj, factor_levels, effects, p_thresh)

adjacency = None
if hemi == 'lh':
    adjacency = mne.spatial_src_adjacency(src[:1]) # lh: src[:1], rh: src[1:]
elif hemi == 'rh':
    adjacency = mne.spatial_src_adjacency(src[1:]) # lh: src[:1], rh: src[1:]
else:
    adjacency = mne.spatial_src_adjacency(src) # both hemispheres, you will need to load in both 'lh' and 'rh' rois separately

In [ ]:
X = data_time # time subset

print("ROI subset: ",roi, '  vertices: ',len(roi_idx)) if use_roi else print(f"Use ROI? {use_roi}")
print("Shape of data: ", X[0].shape)
print("Conditions: ", conditions)
print("Launching clustering test for effect:", effects)
print("Threshold:", f_thresh)

In [ ]:
F_obs, clusters, clusters_pvals, h0 = clu = \
    mne.stats.spatio_temporal_cluster_test(X, 
                                           tail = tail,
                                           threshold = f_thresh,
                                           stat_fun = stat_fun,
                                           n_permutations = n_permutations,
                                           adjacency = adjacency,
                                           spatial_exclude = spatial_exclude,
                                           out_type='indices',
                                           n_jobs = 4)

# Save results
pickle_fname = os.path.join(stats_dir, 
                            'stats_%s_%s-%s_%s.pickled' 
                            % (effects, str(search_tmin), str(search_tmax), label_name))
with open(pickle_fname, "wb") as open_file:
    pickle.dump(clu, open_file)

p_thresh = 0.2      # generous threshold just to see results
print(clusters_pvals)
print(np.where(clusters_pvals < p_thresh)[0])
good_cluster_inds = np.where(clusters_pvals < p_thresh)[0]